# 🧹 Data Cleaning - Online Retail Dataset

In this section, we clean the dataset by handling missing values, removing invalid transactions, and preparing the data for analysis and modeling.

## Importing Libraries and Loading Dataset

We start by importing the required libraries and loading the raw online retail dataset.

In [1]:
import pandas as pd

df = pd.read_csv("../data/online_retail.csv", encoding='ISO-8859-1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Initial Data Inspection

We inspect the dataset structure, data types, and missing values to understand data quality issues before cleaning.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [3]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## Handling Missing Customer Information

CustomerID is essential for identifying purchasing behavior and building personalized recommendations.

Transactions without CustomerID are removed because they cannot be linked to a specific customer.

In [4]:
df = df.dropna(subset=['CustomerID'])

## Removing Cancelled Transactions

Invoices starting with "C" represent cancelled orders.

These transactions do not correspond to real purchases and may negatively affect sales analysis and recommendation quality.

In [5]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

## Removing Invalid Transactions

We remove transactions with:

- Negative quantities (product returns)
- Zero or negative prices

Only valid purchases are kept for accurate business analysis.

In [6]:
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

## Removing Noisy and Non-Product Records

The dataset contains operational records such as:

- Damaged items
- Manual corrections
- Inventory checks
- Mislabelled products

These records do not represent actual customer purchases.

To improve recommendation quality, we keep only valid product codes and remove noisy product descriptions.

In [7]:
# Keep only valid product codes
df = df[df['StockCode'].astype(str).str.len() >= 5]


# Keywords of noisy / non-product entries
trash_keywords = [
    'wrongly',
    'marked',
    'check',
    'damaged',
    'lost',
    'found'
]


# Remove noisy descriptions
df = df[
    ~df['Description'].str.contains(
        '|'.join(trash_keywords),
        case=False,
        na=False,
        regex=True
    )
]

In [8]:
df.shape

(396196, 8)

In [9]:
df['Description'].sample(10)

189248            JUMBO STORAGE BAG SKULLS
149337      HANGING CHICK CREAM DECORATION
222900           LUNCH BAG ALPHABET DESIGN
374543     CHARLOTTE BAG DOLLY GIRL DESIGN
343814    RETROSPOT TEA SET CERAMIC 11 PC 
328271         GIN + TONIC DIET METAL SIGN
198765         CREAM SWEETHEART EGG HOLDER
350091     CLOTHES PEGS RETROSPOT PACK 24 
342273              CARD CHRISTMAS VILLAGE
344450       FRENCH BLUE METAL DOOR SIGN 3
Name: Description, dtype: object

## Converting Date Format

We convert the transaction date into datetime format to enable time-based analysis such as:
- Daily sales
- Monthly trends
- Seasonal behavior

In [10]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

## Removing Duplicate Records

Duplicate transactions may artificially inflate sales metrics and product popularity.

We remove duplicate entries to ensure data consistency.

In [11]:
df = df.drop_duplicates()

## Feature Engineering

Created a new column `TotalPrice` representing total revenue per transaction.

In [12]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

## Final Dataset Verification

We verify the cleaned dataset structure before exporting it for further analysis and modeling.

In [13]:
df.shape

(391009, 9)

In [14]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Exporting Cleaned Dataset

The cleaned dataset is saved and will be used in the next stages:

- Exploratory Data Analysis (EDA)
- Recommendation System Development

In [15]:
df.to_csv("../data/cleaned_retail.csv", index=False)


# Cleaning Summary

During the data cleaning process:

- Removed records with missing `CustomerID`
- Removed cancelled transactions
- Filtered invalid quantities and prices
- Removed noisy and non-product records
- Converted `InvoiceDate` to datetime format
- Removed duplicate records
- Created the `TotalPrice` feature

The dataset is now clean, consistent, and ready for exploratory analysis and recommendation modeling.